# Calculate Transport Scores

Generate three synthetic employee home locations, calculate public-transport commute scores against the project workplace, and save both the employee sample and score outputs.

## 1. Prepare Synthetic Employees

Sample a repeatable, configured number of home points from the population grid within the configured workplace buffer.

In [1]:
from pathlib import Path
import importlib
import sys

project_root = next(
    (path for path in (Path.cwd(), *Path.cwd().parents) if (path / "pyproject.toml").is_file()),
    None,
)
if project_root is None:
    raise RuntimeError("Run this notebook from within the project directory.")
sys.path.insert(0, str(project_root / "src"))

import commute_analysis.config as config
import commute_analysis.geometry as geometry
import commute_analysis.scoring as scoring
import commute_analysis.synthetic_population as synthetic_population

importlib.reload(config)
importlib.reload(geometry)
importlib.reload(synthetic_population)
importlib.reload(scoring)

project_root = config.PROJECT_ROOT
create_target_area = geometry.create_target_area
sample_population_weighted_locations = synthetic_population.sample_population_weighted_locations
calculate_transport_scores_for_geodataframe = scoring.calculate_transport_scores_for_geodataframe
load_transport_scores_from_gpkg = scoring.load_transport_scores_from_gpkg

import geopandas as gpd
import pandas as pd

In [ ]:
target_area = create_target_area(
    *config.WORKPLACE,
    radius_km=config.ANALYSIS_RADIUS_KM,
)

population_df = pd.read_csv(
    config.POPULATION_GRID_PATH,
    sep=config.POPULATION_GRID_READ_SEPARATOR,
    engine=config.POPULATION_GRID_CSV_ENGINE,
)
population_gdf = gpd.GeoDataFrame(
    population_df,
    geometry=gpd.points_from_xy(population_df["x_mp_100m"], population_df["y_mp_100m"]),
    crs=config.CRS_POPULATION_GRID,
)

employees = sample_population_weighted_locations(
    population_grid=population_gdf,
    population_column=config.POPULATION_COLUMN,
    n=config.SYNTHETIC_EMPLOYEE_COUNT,
    target_area=target_area,
    seed=config.RANDOM_SEED,
    filter_mode=config.SAMPLING_FILTER_MODE,
    gaussian_scale=config.SAMPLING_GAUSSIAN_SCALE,
).to_crs(config.CRS_WGS84)

employee_path = config.SCORING_EMPLOYEES_PATH
employee_path.parent.mkdir(parents=True, exist_ok=True)
employees.to_file(
    employee_path,
    layer=config.SYNTHETIC_EMPLOYEES_LAYER,
    driver=config.GEOPACKAGE_DRIVER,
)
employees[["employee_id", "geometry"]]

,employee_id,geometry
0,EMP-00001,POINT (10.10661 53.65483)
1,EMP-00002,POINT (10.0491 53.61263)
2,EMP-00003,POINT (10.0174 53.66925)
3,EMP-00004,POINT (9.9947 53.69801)
4,EMP-00005,POINT (9.9585 53.56501)
...,...,...
95,EMP-00096,POINT (9.94784 53.64048)
96,EMP-00097,POINT (10.08078 53.58835)
97,EMP-00098,POINT (9.83324 53.5658)
98,EMP-00099,POINT (9.94641 53.58028)


## 2. Calculate and Save Scores

The batch scorer evaluates each employee and writes summary metrics and morning route geometries as GeoPackages. The dedicated output folder is overwritten on reruns.

In [ ]:
summary_gdf, route_gdf = calculate_transport_scores_for_geodataframe(
    employee_gdf=employees,
    employee_id_column="employee_id",
    home_column="geometry",
    home_crs=employees.crs,
    max_workers=config.SCORING_MAX_WORKERS,
    save_df=config.SCORING_SAVE_OUTPUTS,
    save_dir=config.SCORING_OUTPUT_DIR,
    overwrite=config.SCORING_OVERWRITE_OUTPUTS,
)

summary_gdf[[
    "employee_id",
    "total_travel_time",
    "transport_score",
    "relative_time_score",
    "absolute_time_score",
    "consistency_score",
    "walking_score",
    "transfers_score",
]]

Processed 1/3 employees
Processed 2/3 employees
Processed 3/3 employees


,employee_id,total_travel_time,transport_score,relative_time_score,absolute_time_score,consistency_score,walking_score,transfers_score
0,EMP-00001,69.716667,0.573516,0.463682,0.612685,0.533543,0.693611,0.750
1,EMP-00002,116.783333,0.445229,0.219377,0.351204,0.720095,0.490556,0.375
2,EMP-00003,52.183333,0.401040,0.110196,0.710093,0.187394,0.522500,0.875


## 3. Verify Saved Results

Reload the saved GeoPackages and confirm that every employee in the configured sample has both summary and route records.

In [ ]:
saved_summary_gdf, saved_route_gdf = load_transport_scores_from_gpkg(
    save_dir=config.SCORING_OUTPUT_DIR,
)
assert len(saved_summary_gdf) == config.SCORING_EMPLOYEE_COUNT
assert len(saved_route_gdf) == config.SCORING_EMPLOYEE_COUNT
saved_summary_gdf[["employee_id", "total_travel_time", "transport_score"]]

,employee_id,total_travel_time,transport_score
0,EMP-00001,69.716667,0.573516
1,EMP-00002,116.783333,0.445229
2,EMP-00003,52.183333,0.401040
